# Stage 2 — subject failure analysis (thesis Figure 2 only)
The original exploratory notebook contained six figures. This version retains only the thesis-relevant intensity-free DRS shape comparison for difficult over- and under-estimated subjects.

In [1]:
from pathlib import Path
import os, sys
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').is_file():  # VS Code may start in notebooks/
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Open this notebook from final_refactored or its notebooks folder.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
from subject_nirs.stage2.subject_failure import (
    load_baseline_performance, load_or_compute_signatures, make_figure2,
)
BASELINE_DIR = Path('artifacts/stage2/baseline/hc')
VALIDATION_CACHE = Path('cache/stage2/preprocessed_cache_grid_50k')
SIGNATURE_CACHE = Path('cache/stage2/subject_failure/subject_mean_signatures.npz')
FIGURE_DIR = Path('results/figures/stage2')

## 1. Build intensity-balanced subject signatures
For every subject, the 30-channel DRS vectors are averaged within each HC × StO₂ target cell and then equally across cells. The compact signature cache is rebuildable and excluded from Git.

In [2]:
subject_ids, signatures = load_or_compute_signatures(
    SIGNATURE_CACHE,
    VALIDATION_CACHE / 'val_x_preprocessed.npy',
    VALIDATION_CACHE / 'val_y_all_targets.npy',
    VALIDATION_CACHE / 'val_subj_0based.npy',
    recompute=False,
)
subject_ids.shape, signatures.shape

((154,), (154, 30))

## 2. Generate Figure 2
Accurate subjects are the lowest baseline-RMSE quartile. Failure groups are the highest-RMSE quartile split by signed bias. Channel contrasts subtract each subject's across-channel mean and use the accurate group MAD as a robust scale.

In [3]:
BASELINE_RESULTS = BASELINE_DIR / 'loso_results.csv'
if not BASELINE_RESULTS.is_file():
    raise FileNotFoundError(f'Run the HC baseline LOSO experiment first. Missing: {BASELINE_RESULTS}')
performance = load_baseline_performance(BASELINE_DIR)
summary = make_figure2(performance, subject_ids, signatures, FIGURE_DIR)
summary

{'accurate_n': 39,
 'overestimation_n': 17,
 'underestimation_n': 22,
 'rmse_q25': 10.521771907806396,
 'rmse_q75': 17.88979196548462}